In [16]:
from openai import OpenAI
import dotenv
dotenv.load_dotenv()
import os

memory__stack = {}

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = os.getenv("NVIDIA_API_KEY")
)

prompt_system = \
"""
task ::
    generate a python code which takes input as document type file path i.e. {{pdf | docx | html | md | xlsx | csv}} and convert into a pdf file

instruction ::
    - strictly ensure to structure the code within the function name `func__code_python(...)` inclusive of imports with takes the `path` as an input argument and save it by calling the fwunction within the generated code itself
    - strictly ensure that the return of the function as to the status code where `0` represents the program converts the input file into pdf file and been saved in to the output path with the same file name but with `.pdf` as extension in the output path `./database/docuAI`, `1` for error and `-1` for warning (even if program execution is success) as save in a variable `memory_stack`
    - strictly ensure the import modules must be cross-platform except for the file formats {{docx | xlsx | (html | md)}} where to use {{docx2pdf | openpyxl | beautifulsoup}} modules respectively
    - strictly ensure to convert whole csv file into pdf file
    - strictly ensure to generate code as per the input file format itself (a single required file format as per the input path file).
    - strictly ensure not to include intro, outro and any code block in the generated response output

input ::
    path : str

output ::
    {{-1, 0, 1}}

code ::
    def func__code_python(path):
        ## install required python modules (using `sys`) ##
        ...
        ## install required python modules (using `sys`) ##

        ## import ##
        ...
        ## import ##

        ## read the input path file ##
        ...
        ## read the input path file ##

        ## process the file into pdf ##
        ...
        ## process the file into pdf ##

        ## write the pdf into output path ##
        ...
        ## write the pdf into output path ##

    return {{-1 | 0 | 1}}

   status = func__code_python(<path>)

"""

prompt_user = \
"""
path : "./warehouse/docuAI/report_2.html"
"""

completion = client.chat.completions.create(
  model="openai/gpt-oss-120b",

  messages=\
    [
        {
            "role":"system",
            "content": prompt_system
        }, 
        {
            "role":"user",
            "content": prompt_user
        }
    ],
  temperature=0.44,
  top_p=0.44,
  max_tokens=4096,
  stream=False
)



In [17]:
print(completion.choices[0].message.content)

def func__code_python(path):
    ## install required python modules (using `sys`) ##
    import sys, subprocess
    def pip_install(pkg):
        try:
            __import__(pkg)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
    pip_install('beautifulsoup4')
    pip_install('reportlab')
    ## install required python modules (using `sys`) ##

    ## import ##
    import os
    from bs4 import BeautifulSoup
    from reportlab.lib.pagesizes import letter
    from reportlab.pdfgen import canvas
    ## import ##

    ## read the input path file ##
    try:
        with open(path, 'r', encoding='utf-8') as f:
            html_content = f.read()
    except Exception as e:
        memory_stack = 1
        return memory_stack
    ## read the input path file ##

    ## process the file into pdf ##
    try:
        soup = BeautifulSoup(html_content, 'html.parser')
        text = soup.get_text(separator='\n')
    except Exception as e

In [18]:
exec(completion.choices[0].message.content, memory__stack)
memory__stack['status']


0

In [19]:
def func__code_python(path):
    import os
    import sys
    import shutil
    import subprocess
    from pathlib import Path

    # Initialize status variable
    memory_stack = 1  # default to error

    try:
        input_path = Path(path).expanduser().resolve()
        if not input_path.is_file():
            raise FileNotFoundError(f"Input file not found: {input_path}")

        # Prepare output directory
        output_dir = Path("./database/docuAI")
        output_dir.mkdir(parents=True, exist_ok=True)

        # Determine output PDF path
        output_pdf = output_dir / (input_path.stem + ".pdf")

        # If the file is already a PDF, just copy it
        if input_path.suffix.lower() == ".pdf":
            shutil.copy2(str(input_path), str(output_pdf))
        else:
            # Use LibreOffice (or OpenOffice) headless conversion if available
            # This works for docx, xlsx, html, md, csv and many other formats
            # The command is cross‑platform as long as the executable is in PATH
            convert_cmd = [
                "soffice", "--headless", "--convert-to", "pdf",
                "--outdir", str(output_dir), str(input_path)
            ]

            # Fallback to "libreoffice" if "soffice" is not found
            if shutil.which("soffice") is None:
                if shutil.which("libreoffice"):
                    convert_cmd[0] = "libreoffice"
                else:
                    raise EnvironmentError("LibreOffice (soffice) not found in PATH.")

            result = subprocess.run(convert_cmd, capture_output=True, text=True)

            if result.returncode != 0:
                raise RuntimeError(f"Conversion failed: {result.stderr}")

            # LibreOffice names the output file with the same stem and .pdf extension
            # Ensure the expected file exists
            if not output_pdf.is_file():
                # LibreOffice may place the file directly in output_dir with original name
                possible = output_dir / (input_path.stem + ".pdf")
                if possible.is_file():
                    output_pdf = possible
                else:
                    raise FileNotFoundError("Converted PDF not found after LibreOffice run.")

        # If we reach this point, conversion succeeded
        memory_stack = 0

        # Optional warning: if the original file size is zero, flag a warning
        if input_path.stat().st_size == 0:
            memory_stack = -1

    except Exception as e:
        # Log the error to stderr (could be replaced with proper logging)
        sys.stderr.write(f"Error: {e}\n")
        memory_stack = 1

    return memory_stack


In [20]:
!.\.venv\Scripts\Activate.ps1

In [22]:
from pathlib import Path
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling_core.types.doc.document import DoclingDocument, ContentLayer
from docling_core.types.doc import SectionHeaderItem, TextItem, TableItem, PictureItem
from docling_core.transforms.visualizer.layout_visualizer import LayoutVisualizer
import os


pipeline_options = PdfPipelineOptions(generate_page_images=True, images_scale=1.0)
converter = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)})
visualizer = LayoutVisualizer()
visualizer.params.show_label = False

lst_pdf = [os.path.splitext(itr)[0] for itr in os.listdir("./database/docuAI/phase_01/step_01")]
for itr in lst_pdf:
    result = converter.convert(f"./database/docuAI/phase_01/step_01/{itr}.pdf")
    doc = result.document


    filtered_doc = DoclingDocument(name=f"{doc.name}_level1")
    for page_no, page_item in doc.pages.items():
        filtered_doc.pages[page_no] = page_item
    for item, level in doc.iterate_items(included_content_layers={ContentLayer.BODY}):
        if level != 1:
            continue
        prov = item.prov[0] if getattr(item, "prov", None) else None
        if isinstance(item, SectionHeaderItem):
            filtered_doc.add_heading(text=item.text, level=item.level, prov=prov)
        elif isinstance(item, TableItem):
            filtered_doc.add_table(data=item.data, prov=prov)
        elif isinstance(item, PictureItem):
            filtered_doc.add_picture(prov=prov)
        elif isinstance(item, TextItem):
            filtered_doc.add_text(label=item.label, text=item.text, prov=prov)


    images = visualizer.get_visualization(doc=filtered_doc)
    for page_num, pil_image in images.items():
        output_path = Path(f"./database/docuAI/phase_01/step_02/{itr}/page_{page_num:0{len(str(len(images)))}d}.png")
        output_path.parent.mkdir(parents=True, exist_ok=True)
        pil_image.save(output_path)

[INFO] 2026-07-04 16:48:36,441 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-04 16:48:36,446 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-04 16:48:36,485 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\shrey\OneDrive\Desktop\docuAI\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-07-04 16:48:36,489 [RapidOCR] main.py:50: Using C:\Users\shrey\OneDrive\Desktop\docuAI\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-07-04 16:48:36,874 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-04 16:48:36,879 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-04 16:48:36,884 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\shrey\OneDrive\Desktop\docuAI\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-07-04 16:48:36,886 [RapidOCR] main.py:50: Using C:\Users\shrey\OneDrive\Desktop\docuAI\.venv\Lib\site-packages\ra

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

In [2]:
import os
import requests
import base64
import json
import time
from PIL import Image
import io

api_key = os.getenv("NVIDIA_API_KEY")
if not api_key:
    raise RuntimeError("NVIDIA_API_KEY not set")

header_auth = f"Bearer {api_key}"
invoke_url = "https://integrate.api.nvidia.com/v1/chat/completions"
folder = "./database/docuAI/phase_01/step_02/Stress-Dossier_Synthetic-Data"
doc_file_name = "Stress-Dossier_Synthetic-Data"

all_pages = []  # list format now, not dict

def encode_image_for_budget(image_path, max_b64_chars=42000, max_dim=2000):
    img = Image.open(image_path).convert("L")
    quality = 75
    dim = max_dim

    while True:
        img_resized = img.copy()
        img_resized.thumbnail((dim, dim))
        buf = io.BytesIO()
        img_resized.save(buf, format="JPEG", quality=quality)
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

        if len(b64) <= max_b64_chars:
            return b64, dim, quality

        if quality > 40:
            quality -= 10
        else:
            dim = int(dim * 0.85)

        if dim < 500:
            return b64, dim, quality

def call_inference(image_b64, prompt_text, retries=3):
    payload = {
        "model": "meta/llama-3.2-90b-vision-instruct",
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt_text},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}
                    }
                ]
            }
        ],
        "max_tokens": 1200,
        "temperature": 0.44,
        "top_p": 0.44,
        "stream": False
    }
    headers = {
        "Authorization": header_auth,
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    for attempt in range(retries):
        try:
            resp = requests.post(invoke_url, headers=headers, json=payload, timeout=(10, 120))
            return resp
        except requests.exceptions.ReadTimeout:
            wait = 10 * (attempt + 1)
            print(f"    read timeout, retrying in {wait}s")
            time.sleep(wait)
    return None

prompt_text = """
task: analyze this document page image and extract structured metadata.

instruction:
- respond with ONLY raw json, no intro, no outro, no markdown code block
- transcribe exact codes, reference numbers, dates, and titles VERBATIM as they appear on the page (do not paraphrase numbers/codes)
- describe every element on the page: titles, tables (with column names), signatures, headers, footers
- if a value is unreadable, write "unclear" rather than guessing

output format (single json object, not a list):
{
  "description": "<exhaustive description of every element on the page, including exact titles and table structure>",
  "reference": "<all exact reference codes, document numbers, dates, issue numbers found on the page, verbatim>"
}
"""

png_files = sorted([f for f in os.listdir(folder) if f.endswith(".png")])
print(f"found {len(png_files)} pages")

for i, filename in enumerate(png_files, 1):
    page_number = int(filename.replace("page_", "").replace(".png", ""))
    image_path = os.path.join(folder, filename)
    print(f"[{i}/{len(png_files)}] {filename}")

    image_b64, dim, quality = encode_image_for_budget(image_path)
    print(f"    encoded: {dim}px q{quality}, {len(image_b64)} b64 chars")

    resp = call_inference(image_b64, prompt_text)

    if resp is None:
        print(f"    FAILED — gave up after retries")
        continue
    if resp.status_code != 200:
        print(f"    HTTP {resp.status_code}: {resp.text[:300]}")
        continue

    try:
        response = resp.json()
        raw_content = response["choices"][0]["message"]["content"]
    except (json.JSONDecodeError, KeyError) as e:
        print(f"    bad response structure: {e}")
        continue

    try:
        page_data = json.loads(raw_content)
        all_pages.append({
            "file_name": doc_file_name,
            "page_no": page_number,
            "description": page_data.get("description", ""),
            "reference": page_data.get("reference", "")
        })
        print(f"    OK")
    except json.JSONDecodeError:
        print(f"    model output not valid JSON: {raw_content[:300]}")
        continue

print(f"\ndone — {len(all_pages)}/{len(png_files)} pages extracted")
print(json.dumps(all_pages, indent=2))

found 2 pages
[1/2] page_1.png
    encoded: 640px q35, 39488 b64 chars
    read timeout, retrying in 10s
    OK
[2/2] page_2.png
    encoded: 544px q35, 31220 b64 chars
    read timeout, retrying in 10s
    read timeout, retrying in 20s
    OK

done — 2/2 pages extracted
[
  {
    "file_name": "Stress-Dossier_Synthetic-Data",
    "page_no": 1,
    "description": "This document is a technical report titled 'STRUT HOUSING KINETIC STRESS ANALYSIS & JUSTIFICATION DATAPACK (POST-MOD 45882 AND PRE-MOD 77291)'. It features a table with columns labeled 'ATA CHAPTER', '20 HYDRAULIC SYSTEMS', 'A/C SECTION', '42 INTEGRATED STRUT ASSEMBLY', 'A/C PART ASSEMBLY', 'PYLON MOUNT BRACKETS', 'A/C PART', 'FORWARD LUG', 'ENGINE APPLICABILITY', 'RE3-8001 GEn-38 TURBOFAN', 'ANALYSIS TYPE', 'THERMAL STRESS AND VIBRATION ANALYSIS', and 'A/C APPLICABILITY'. The table contains data for each column, including 'ATA CHAPTER' values such as '20 HYDRAULIC SYSTEMS' and 'A/C SECTION' values like '42 INTEGRATED STRUT AS

In [16]:
import os
import json
import requests
from neo4j import GraphDatabase
from dotenv import load_dotenv

# ---- Load environment ----
load_dotenv()  # if this doesn't pick up your .env, pass dotenv_path="C:/full/path/.env"

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")


invoke_url = "https://integrate.api.nvidia.com/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {NVIDIA_API_KEY}",
    "Accept": "application/json"
}
prompt_system = """
You are given a document represented as multiple pages.

Return ONLY valid JSON.
Do NOT return markdown.
Do NOT explain anything.

Output Schema:

{
  "pages": [
    {
      "page_no": 1,
      "description": "...",
      "reference": "..."
    }
  ],
  "references": [
    {
      "from_page": 2,
      "to_page": 5,
      "reference_contains": "..."
    }
  ]
}

Instructions:

1. Copy every page exactly as provided into the "pages" array.

2. Compare EVERY page against EVERY other page.

3. Determine whether a page references another page using BOTH:
   - description
   - reference

4. A reference may be:
   - a table
   - a figure
   - a section
   - a heading
   - a topic
   - a procedure
   - a semantic description
   - a document element

5. Match semantically, not only exact words.

6. If Page A clearly refers to content explained on Page B,
   create

{
    "from_page": A,
    "to_page": B,
    "reference_contains": "<text that caused the match>"
}

7. Never invent page numbers.

8. Never create self references.

9. Multiple references from one page are allowed.

10. If no reference exists,
return

"references": []

Return ONLY JSON.
"""
def func__code_python(all_pages, document_name="AEROCORP"):

    payload = {
        "model": "openai/gpt-oss-120b",
        "messages": [
            {
                "role": "system",
                "content": prompt_system
            },
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "document_name": document_name,
                        "pages": all_pages
                    },
                    indent=2
                )
            }
        ],
        "temperature": 0.1,
        "top_p": 0.9,
        "max_tokens": 8192,
        "stream": False
    }

    response = requests.post(
        invoke_url,
        headers=headers,
        json=payload
    )

    if response.status_code != 200:
        print("API Error:", response.status_code)
        print(response.text)
        return -1

    response = response.json()

    if "choices" not in response:
        print("No response from model.")
        return -1

    content = response["choices"][0]["message"]["content"].strip()

    if content.startswith("```"):
        content = (
            content.replace("```json", "")
                   .replace("```", "")
                   .strip()
        )

    try:
        data = json.loads(content)
    except Exception as e:
        print("JSON Parse Error:", e)
        print(content)
        return -1

    pages = data.get("pages", [])
    references = data.get("references", [])

    print(f"Pages: {len(pages)}")
    print(f"References: {len(references)}")

    with driver.session() as session:

        session.run(
            """
            MERGE (d:Document {document_name:$document_name})
            """,
            document_name=document_name
        )

        for p in pages:

            session.run(
            """
            MATCH (d:Document {document_name:$document_name})

            MERGE (pg:Page {
                page_no:$page_no,
                document_name:$document_name
            })

            SET pg.name = "Page " + toString($page_no),
                pg.description = $description,
                pg.reference = $reference

            MERGE (d)-[:HAS_PAGE]->(pg)
            """,
            document_name=document_name,
            page_no=p["page_no"],
            description=p.get("description", ""),
            reference=p.get("reference", "")
        )
        for r in references:

                session.run(
        """
        MATCH (a:Page {
            page_no:$from_page,
            document_name:$document_name
        })

        MATCH (b:Page {
            page_no:$to_page,
            document_name:$document_name
        })

        MERGE (a)-[rel:REFERENCES]->(b)

        SET rel.reference_contains=$reference_contains
        """,
        document_name=document_name,
        from_page=r["from_page"],
        to_page=r["to_page"],
        reference_contains=r.get("reference_contains", "")
    )

    print("Graph Loaded Successfully.")
    return 0
if __name__ == "__main__":
    status = func__code_python(
    all_pages,
    document_name="Stress-Dossier_Synthetic-Data"
)

print(status)
driver.close()


Pages: 2
References: 0
Graph Loaded Successfully.
0


In [15]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

driver.verify_connectivity()
print("Connected successfully!")

Connected successfully!


In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
print(NEO4J_URI)
print(NEO4J_USER)

neo4j+s://1f255d87.databases.neo4j.io
1f255d87


In [9]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os
import json
import re
import requests
import time

load_dotenv()

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

# ---------------------------------------------------------
# Step 1 : Fetch all pages from Neo4j
# ---------------------------------------------------------
def get_all_pages():

    query = """
  MATCH (d:Document)-[:HAS_PAGE]->(p:Page)
RETURN
    d.document_name AS file_name,
    p.page_no AS page_no,
    p.description AS description,
    p.reference AS reference
ORDER BY p.page_no
    """

    with driver.session() as session:

        result = session.run(query)

        return [
            {
                "file_name": record["file_name"],
                "page_no": record["page_no"],
                "description": record["description"]
            }
            for record in result
        ]


# ---------------------------------------------------------
# Step 2 : Ask LLM to identify matching pages
# ---------------------------------------------------------
def get_matching_pages_from_llm(query, pages):

    context = ""

    for page in pages:

        context += f"""
File Name: {page['file_name']}
Page Number: {page['page_no']}

Description:
{page['description']}

Reference:
{page.get("reference", "")}

-----------------------------------------
"""

    user_prompt = f"""
Below are page descriptions extracted from a document.

{context}

User Question:
{query}

Return ONLY valid JSON.

{{
  "results":[
    {{
      "file_name":"Stress-Dossier_Synthetic-Data",
      "page_no":2
    }}
  ]
}}
"""

    system_prompt = """
You are a semantic page retrieval assistant.
The page description may summarize text, tables, figures and images.

If the user's question refers to a value contained in a table,
figure, appendix, or reference, use both the description and
reference fields to identify the correct page.

Return the closest matching page even if the wording is not exactly
the same.


Return ONLY JSON.
"""

    invoke_url = "https://integrate.api.nvidia.com/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {NVIDIA_API_KEY}",
        "Accept": "application/json",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "openai/gpt-oss-120b",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0,
        "max_tokens": 300
    }

    print("\nSending request to NVIDIA...\n")

    start = time.time()

    response = requests.post(
        invoke_url,
        headers=headers,
        json=payload,
        timeout=300
    )

    elapsed = time.time() - start

    print(f"Time Taken : {elapsed:.2f} seconds")
    print("Status :", response.status_code)

    response.raise_for_status()

    result = response.json()

    message = result["choices"][0]["message"]

    raw = message.get("content") or message.get("reasoning_content") or ""

    return parse_llm_json(raw)


# ---------------------------------------------------------
# Step 3 : Parse JSON
# ---------------------------------------------------------
def parse_llm_json(raw):

    cleaned = raw.strip()

    cleaned = re.sub(r"^```(?:json)?", "", cleaned)
    cleaned = re.sub(r"```$", "", cleaned)

    try:

        parsed = json.loads(cleaned)

        return parsed.get("results", [])

    except:

        match = re.search(r"\{.*\}", cleaned, re.DOTALL)

        if match:

            parsed = json.loads(match.group())

            return parsed.get("results", [])

        raise ValueError("Cannot parse LLM JSON.")


# ---------------------------------------------------------
# Step 4 : Retrieve matched page + referenced pages
# ---------------------------------------------------------
def get_pages_by_matches(matches):

    fetched = []

    query = """
    MATCH (d:Document)-[:HAS_PAGE]->(p:Page)
    WHERE d.document_name = $file_name
      AND p.page_no = $page_no

    OPTIONAL MATCH (p)-[r:REFERENCES]->(ref:Page)

    RETURN

        d.document_name AS file_name,

        p.page_no AS page_no,

        p.description AS description,

        collect(
            CASE
            WHEN ref IS NULL THEN NULL
            ELSE {

                page_no: ref.page_no,

                description: ref.description,

                reference_contains: r.reference_contains

            }
            END
        ) AS references
    """

    with driver.session() as session:

        for match in matches:

            result = session.run(
                query,
                file_name=match["file_name"],
                page_no=match["page_no"]
            )

            for record in result:

                refs = []

                for r in record["references"]:

                    if r is not None:

                        refs.append(r)

                fetched.append({

                    "file_name": record["file_name"],

                    "page_no": record["page_no"],

                    "description": record["description"],

                    "references": refs

                })

    return fetched


# ---------------------------------------------------------
# Step 5 : Main
# ---------------------------------------------------------
def answer_query(query):

    pages = get_all_pages()

    print("Pages Found :", len(pages))

    matches = get_matching_pages_from_llm(query, pages)

    print("\nLLM Matches\n")

    print(json.dumps(matches, indent=2))

    if not matches:

        print("No match found.")

        return []

    retrieved = get_pages_by_matches(matches)

    print("\nRetrieved Result\n")

    print(json.dumps(retrieved, indent=2))

    return retrieved


# ---------------------------------------------------------
# Run
# ---------------------------------------------------------
if __name__ == "__main__":

    query = "What is the RF value described at a given location in the structural detail?"

    answer_query(query)

    driver.close()

Pages Found : 2

Sending request to NVIDIA...

Time Taken : 2.89 seconds
Status : 200

LLM Matches

[
  {
    "file_name": "Stress-Dossier_Synthetic-Data",
    "page_no": 1
  }
]

Retrieved Result

[
  {
    "file_name": "Stress-Dossier_Synthetic-Data",
    "page_no": 1,
    "description": "The document is titled 'AEROCORP' and contains a report on 'STRUT HOUSING KINETIC STRESS ANALYSIS & JUSTIFICATION DATAPAP (POST-MOD 45B82 AND PRE-MOD 77291)'. It includes details such as origin, reference, issue, date, and number of pages. The report covers various sections like ATA Chapter, A/C Section, A/C Part Assembly, Engine Applicability, Analysis Type, and A/C Applicability.",
    "references": []
  }
]


In [9]:
import os
import requests
import base64
import json
import time
import re
from PIL import Image
import io

api_key = os.getenv("NVIDIA_API_KEY")
header_auth = f"Bearer {api_key}"
invoke_url = "https://integrate.api.nvidia.com/v1/chat/completions"
folder = "./database/docuAI/phase_01/step_02/Stress-Dossier_Synthetic-Data"

def strip_json_fence(text):
    """Model sometimes wraps output in ```json ... ``` — strip it before parsing."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


# ---------- STAGE 1: retrieve matching page(s) from descriptions ----------

def retrieve_matching_pages(query, all_pages, top_k=3, retries=3):
    """all_pages: list of {file_name, page_no, description, reference} — no images, text-only match."""
    index_text = "\n".join(
        f"page_no: {p['page_no']} | description: {p['description']} | reference: {p['reference']}"
        for p in all_pages
    )

    prompt = f"""
task: find the most relevant page(s) for the user query, based on the page index below.

instruction:
- respond with ONLY raw json, no intro, no outro, no markdown code block
- return up to {top_k} page numbers, ordered by relevance (most relevant first)
- if nothing is relevant, return an empty list

output format:
{{"matching_pages": [<page_no>, <page_no>, ...]}}

user query: "{query}"

page index:
{index_text}
"""

    payload = {
        "model": "openai/gpt-oss-120b",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 300,
        "temperature": 0.2,
        "top_p": 0.7,
        "stream": False
    }
    headers = {"Authorization": header_auth, "Content-Type": "application/json", "Accept": "application/json"}

    for attempt in range(retries):
        try:
            resp = requests.post(invoke_url, headers=headers, json=payload, timeout=(10, 60))
        except requests.exceptions.ReadTimeout:
            print(f"  retrieval timeout, retry {attempt+1}/{retries}")
            time.sleep(5 * (attempt + 1))
            continue

        if resp.status_code != 200:
            print(f"  retrieval HTTP {resp.status_code}: {resp.text[:300]}")
            return []

        raw_content = resp.json()["choices"][0]["message"]["content"]
        print(f"  raw model output: {raw_content[:300]}")  # log before parsing — your earlier debug point

        try:
            parsed = json.loads(strip_json_fence(raw_content))
            return parsed.get("matching_pages", [])
        except json.JSONDecodeError:
            print(f"  retrieval output not valid JSON: {raw_content[:300]}")
            return []

    return []


# ---------- STAGE 2: answer query grounded on the retrieved page image ----------

def encode_full_page(image_path, max_b64_chars=42000, max_dim=2000):
    img = Image.open(image_path).convert("L")
    quality = 75
    dim = max_dim
    while True:
        img_resized = img.copy()
        img_resized.thumbnail((dim, dim))
        buf = io.BytesIO()
        img_resized.save(buf, format="JPEG", quality=quality)
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
        if len(b64) <= max_b64_chars:
            return b64
        if quality > 40:
            quality -= 10
        else:
            dim = int(dim * 0.85)
        if dim < 500:
            return b64


def answer_query_from_page(query, page_no, retries=3):
    filename = f"page_{page_no}.png"
    image_path = os.path.join(folder, filename)

    if not os.path.exists(image_path):
        return f"page {page_no} image not found at {image_path}"

    image_b64 = encode_full_page(image_path)

    prompt = f"""
task: answer the user's query using ONLY the content visible in this page image.
instruction: be precise, quote exact values/codes/numbers as they appear, INCLUDING UNITS (kg, tonnes, mm, etc). if the answer isn't on this page, say so.
user query: "{query}"
"""

    payload = {
        "model": "meta/llama-3.2-90b-vision-instruct",
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}}
                ]
            }
        ],
        "max_tokens": 800,
        "temperature": 0.3,
        "top_p": 0.7,
        "stream": False
    }
    headers = {"Authorization": header_auth, "Content-Type": "application/json", "Accept": "application/json"}

    for attempt in range(retries):
        try:
            resp = requests.post(invoke_url, headers=headers, json=payload, timeout=(10, 120))
        except requests.exceptions.ReadTimeout:
            print(f"  answer timeout, retry {attempt+1}/{retries}")
            time.sleep(10 * (attempt + 1))
            continue

        if resp.status_code != 200:
            return f"HTTP {resp.status_code}: {resp.text[:300]}"

        return resp.json()["choices"][0]["message"]["content"]

    return "gave up after retries"


# ---------- FULL FLOW ----------

def rag_query(query, all_pages):
    print(f"query: {query}")
    matched_pages = retrieve_matching_pages(query, all_pages)
    print(f"matched pages: {matched_pages}")

    if not matched_pages:
        return {"query": query, "matched_pages": [], "answer": "no relevant page found"}

    results = []
    for page_no in matched_pages:
        answer = answer_query_from_page(query, page_no)
        results.append({"page_no": page_no, "answer": answer})

    return {"query": query, "matched_pages": matched_pages, "results": results}



result = rag_query("what is the MTOW for MP-D450neo?", all_pages)
print(json.dumps(result, indent=2))

query: what is the MTOW for MP-D450neo?
  raw model output: {"matching_pages": [2]}
matched pages: [2]
{
  "query": "what is the MTOW for MP-D450neo?",
  "matched_pages": [
    2
  ],
  "results": [
    {
      "page_no": 2,
      "answer": "The MTOW for MP-D450neo is 78.9 tonnes."
    }
  ]
}
